In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, lit, when, dayofweek, date_format, year, quarter, month, dayofmonth, weekofyear, current_timestamp
from pyspark.sql.types import StructType, StructField, DateType, IntegerType, StringType, BooleanType
import pandas as pd
from datetime import datetime, timedelta

print("CREATING DATE DIMENSION")

gold_date_dim_path = "s3://travel-analytics-bronze/delta/gold/Dim_Date/"

# =============================================================
# STEP 1: GENERATE DATE RANGE (2009-2030)
# =============================================================
print("\nSTEP 1: Generating Date Range from 2009 to 2030...")

# Define date range
start_date = "2009-01-01"
end_date = "2030-12-31"

# Create date range using pandas
date_range = pd.date_range(start=start_date, end=end_date, freq='D')
print(f"Generating {len(date_range):,} date records from {start_date} to {end_date}")

# Create a list of date data
date_data = []
for date in date_range:
    date_str = date.strftime('%Y-%m-%d')
    year_num = date.year
    quarter_num = (date.month - 1) // 3 + 1
    month_num = date.month
    month_name = date.strftime('%B')
    day_of_month = date.day
    day_of_week_num = date.weekday() + 1  # Monday=1, Sunday=7
    weekday_name = date.strftime('%A')
    week_num = date.isocalendar()[1]  # Week number
    
    # Determine if weekend (Saturday=6, Sunday=7)
    is_weekend = 1 if day_of_week_num in [6, 7] else 0
    
    # Basic holiday detection (you can expand this with more holidays)
    is_holiday = 0
    # New Year's Day
    if date.month == 1 and date.day == 1:
        is_holiday = 1
    # Christmas
    elif date.month == 12 and date.day == 25:
        is_holiday = 1
    # Independence Day (US - July 4th)
    elif date.month == 7 and date.day == 4:
        is_holiday = 1
    
    date_data.append((date_str, day_of_month, weekday_name, week_num, month_num, 
                     month_name, quarter_num, year_num, is_weekend, is_holiday))

# Create DataFrame from the generated data
date_schema = StructType([
    StructField("Full_Date", StringType(), False),
    StructField("Day", IntegerType(), False),
    StructField("Weekday_Name", StringType(), False),
    StructField("Week", IntegerType(), False),
    StructField("Month", IntegerType(), False),
    StructField("Month_Name", StringType(), False),
    StructField("Quarter", IntegerType(), False),
    StructField("Year", IntegerType(), False),
    StructField("Is_Weekend", IntegerType(), False),
    StructField("Is_Holiday", IntegerType(), False)
])

date_df = spark.createDataFrame(date_data, schema=date_schema)

# =============================================================
# STEP 2: ADD SURROGATE KEY
# =============================================================
print("\nSTEP 2: Creating Dimension Structure")
dim_date_df = (
    date_df
    
    # Add surrogate key
    .withColumn("Dim_Date_SK", monotonically_increasing_id() + 1)

    # Reorder columns to match dimension model
    .select(
        col("Dim_Date_SK"),           # PK
        col("Full_Date"),
        col("Day"),
        col("Weekday_Name"),
        col("Week"),
        col("Month"),
        col("Month_Name"),
        col("Quarter"),
        col("Year"),
        col("Is_Weekend"),
        col("Is_Holiday")
    )
)


CREATING DATE DIMENSION

STEP 1: Generating Date Range from 2009 to 2030...
Generating 8,035 date records from 2009-01-01 to 2030-12-31

STEP 2: Creating Dimension Structure


In [0]:
dim_date_df.display()

Dim_Date_SK,Full_Date,Day,Weekday_Name,Week,Month,Month_Name,Quarter,Year,Is_Weekend,Is_Holiday
1,2009-01-01,1,Thursday,1,1,January,1,2009,0,1
2,2009-01-02,2,Friday,1,1,January,1,2009,0,0
3,2009-01-03,3,Saturday,1,1,January,1,2009,1,0
4,2009-01-04,4,Sunday,1,1,January,1,2009,1,0
5,2009-01-05,5,Monday,2,1,January,1,2009,0,0
6,2009-01-06,6,Tuesday,2,1,January,1,2009,0,0
7,2009-01-07,7,Wednesday,2,1,January,1,2009,0,0
8,2009-01-08,8,Thursday,2,1,January,1,2009,0,0
9,2009-01-09,9,Friday,2,1,January,1,2009,0,0
10,2009-01-10,10,Saturday,2,1,January,1,2009,1,0


In [0]:
# =============================================================
# STEP 3: SAVE TO GOLD LAYER
# =============================================================
print("\nSTEP 3: Saving to Gold Layer...")
dim_date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_date_dim_path)

print(f"Successfully created Dim_Date with {dim_date_df.count():,} records")
print(f"Date range: {start_date} to {end_date}")


STEP 3: Saving to Gold Layer...
Successfully created Dim_Date with 8,035 records
Date range: 2009-01-01 to 2030-12-31
